# 03 - MACE Training
Train the message-passing MACE model with the same training setup as ACE and model-specific architecture choices.


In [ ]:
import sys
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.loader import DataLoader
import pandas as pd

torch.manual_seed(42)

sys.path.append("..")
from src.dataset import MDTrajectoryDataset
from src.models.mace_wrapper import MACEWrapper
from src.trainer import BenchmarkTrainer


In [ ]:
# Load Data
train_ds = MDTrajectoryDataset("../data/train.extxyz", cutoff=5.0)
val_ds = MDTrajectoryDataset("../data/val.extxyz", cutoff=5.0)
test_ds = MDTrajectoryDataset("../data/test.extxyz", cutoff=5.0)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)


In [ ]:
# Initialize MACE model (shared training setup, model-specific architecture)
model = MACEWrapper(
    num_elements=120,
    r_max=5.0,
    num_radial=8,
    l_max=2,
    num_blocks=2,  # 2 layers of message passing
    node_dim=16
)

optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

trainer = BenchmarkTrainer(
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    train_loader=train_loader,
    val_loader=val_loader,
    device="cuda" if torch.cuda.is_available() else "cpu",
    energy_weight=1.0,
    force_weight=100.0
)


In [ ]:
# Train + held-out test evaluation
metrics_df = trainer.train(max_epochs=50, patience=10)
metrics_df.to_csv("../data/mace_metrics.csv", index=False)

test_metrics = trainer.test_epoch(test_loader)
mace_test_df = pd.DataFrame([test_metrics])
mace_test_df.to_csv("../data/mace_test_metrics.csv", index=False)

metrics_df.head()